# Exoplanetas con datos reales

Ejecuta las celdas **en orden**, con el botón ▶ de cada una.

1. **Paso 1:** descarga los datos de las estrellas.
2. **Paso 2:** escribe el nombre de tu estrella y mira su curva de luz.
3. Sigue la guía rápida que aparece, anota los valores en tu tabla y haz los cálculos con tu calculadora.

## Paso 1 · Cargar los datos

Al ejecutar la celda, los datos se descargan solos desde GitHub. Si no se pudiera, aparecerá un botón para subir el `.zip` (o los `.csv`) que te dio tu maestro. Si cierras el cuaderno, tendrás que repetir este paso.

In [ ]:
# PASO 1 · Cargar los datos (se hace una sola vez)
import io
import shutil
import subprocess
import zipfile
from pathlib import Path

REPO = "https://github.com/BreezyOF/explorando-transitos.git"  # el maestro cambia BreezyOF
CARPETA = Path("datos")


def listar_csv():
    """Archivos .csv de las estrellas (ignora los archivos ocultos que crean algunos .zip)."""
    return sorted(
        a for a in CARPETA.rglob("*.csv")
        if not a.name.startswith(".") and "__MACOSX" not in a.parts
    )


def descargar_del_repositorio():
    """Copia la carpeta datos/ desde GitHub. Devuelve True si funcionó."""
    temporal = Path("_repo")
    shutil.rmtree(temporal, ignore_errors=True)
    resultado = subprocess.run(["git", "clone", "--depth", "1", REPO, str(temporal)],
                               capture_output=True, text=True)
    if resultado.returncode != 0 or not (temporal / "datos").is_dir():
        shutil.rmtree(temporal, ignore_errors=True)
        return False
    shutil.copytree(temporal / "datos", CARPETA, dirs_exist_ok=True)
    shutil.rmtree(temporal, ignore_errors=True)
    return True


CARPETA.mkdir(exist_ok=True)
if descargar_del_repositorio():
    print("Datos descargados desde GitHub.")
else:
    # Plan B: subir los archivos a mano
    from google.colab import files
    print("No se pudo descargar desde GitHub.")
    print("Selecciona el archivo .zip (o los .csv) que te dio tu maestro:")
    subido = files.upload()
    for nombre_archivo, contenido in subido.items():
        if nombre_archivo.lower().endswith(".zip"):
            zipfile.ZipFile(io.BytesIO(contenido)).extractall(CARPETA)
        elif nombre_archivo.lower().endswith(".csv"):
            (CARPETA / nombre_archivo).write_bytes(contenido)

estrellas = [a.stem for a in listar_csv()]
if estrellas:
    print("\nEstrellas disponibles:", ", ".join(estrellas))
else:
    print("\nNo se encontró ningún archivo .csv. Avisa a tu maestro.")

## Paso 2 · Elige tu estrella y mira la curva

Escribe el nombre de tu estrella cuando el programa te lo pida. Para trabajar con otra estrella, vuelve a ejecutar esta celda.

In [ ]:
# PASO 2 · Elige tu estrella y mira su curva de luz
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display


def normalizar(texto):
    """'HD 209458', 'hd-209458' y 'HD209458' se consideran iguales."""
    return re.sub(r"[^a-z0-9]", "", texto.lower())


def claves(archivo):
    """Nombres con los que se puede pedir 'ESTRELLA_planeta_CATALOGO.csv'."""
    partes = archivo.stem.split("_")
    return {normalizar("".join(partes[:n])) for n in range(1, len(partes) + 1)}


def elegir_estrella():
    archivos = listar_csv()
    if not archivos:
        raise FileNotFoundError("No hay archivos .csv. Ejecuta primero la celda del PASO 1.")
    while True:
        texto = input("Escribe el nombre de la estrella (ej. HD 209458): ").strip()
        buscado = normalizar(texto)
        coincidencias = [a for a in archivos if buscado and buscado in claves(a)]
        if len(coincidencias) == 1:
            return coincidencias[0]
        if coincidencias:
            print("Hay varios archivos para esa estrella:")
            for n, a in enumerate(coincidencias, 1):
                print(f"  {n}. {a.stem}")
            r = input("Elige un número: ").strip()
            if r.isdigit() and 1 <= int(r) <= len(coincidencias):
                return coincidencias[int(r) - 1]
            continue
        print(f"'{texto}' no está. Archivos disponibles: {', '.join(a.stem for a in archivos)}")


archivo = elegir_estrella()
nombre = archivo.stem.replace("_", " ")

datos = pd.read_csv(archivo)
time = datos["tiempo"].values
flux = datos["flujo"].values
flux_suave = pd.Series(flux).rolling(15, center=True, min_periods=1).median().values

display(Markdown(f"""
---
### Guía rápida · {nombre}

**Radio del planeta**
1. Pasa el cursor sobre varios puntos **antes y después** del tránsito (brillo normal) y anota su tiempo y su flujo.
2. Haz lo mismo con varios puntos **dentro** del tránsito (la parte más baja).
3. Promedia cada grupo: **F_fuera** (antes y después) y **F_dentro**.
4. Calcula:
   - profundidad = (F_fuera − F_dentro) / F_fuera
   - Rp / R★ = raíz cuadrada(profundidad)
   - Rp = (Rp / R★) × R★  (R★ = radio de la estrella: investígalo y anota la fuente)

**Período**
1. Anota el tiempo del **centro** de varios tránsitos seguidos.
2. Período = tiempo del centro 2 − tiempo del centro 1 (con varios, promedia las diferencias).

**La gráfica**
- Pasa el cursor sobre un punto para leer su tiempo y su flujo.
- Arrastra un rectángulo para acercarte. Doble clic para volver a la vista completa.

---
"""))

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=time, y=flux, mode="markers", name="Flujo normalizado",
    marker=dict(size=4, opacity=0.45),
    hovertemplate="tiempo = %{x:.4f} d<br>flujo = %{y:.5f}<extra></extra>",
))
fig.add_trace(go.Scattergl(
    x=time, y=flux_suave, mode="lines", name="Flujo suavizado (solo guía)",
    line=dict(color="red", width=1.5), hoverinfo="skip",
))
fig.update_layout(
    title=f"{nombre} · pasa el cursor sobre un punto para leer su tiempo y su flujo",
    xaxis_title="Tiempo (días)",
    yaxis_title="Flujo normalizado",
    hovermode="closest",
    hoverdistance=40,
    height=520,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.2),
)
fig.update_xaxes(showspikes=True, spikemode="across", spikethickness=1)
fig.show(config={"displaylogo": False})